<div style="text-align: right">Cesar Aguirre, 04/05/2024</div>
<div style="text-align: right">Let's hope the page has not changed its structure since </div>

# UEFA Webscraping


In [1]:
# Let's start with the necessary imports

import re
import time
import os, sys
import unidecode
import numpy as np
import pandas as pd
import more_itertools as mit

from tqdm import tqdm
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.keys import Keys

In [2]:
# Set up Selenium WebDriver for Chrome
options = webdriver.ChromeOptions()
# options.add_argument('--headless')

# Set up the driver
driver = webdriver.Chrome(options=options)

# Navigate to the website
year_of_interest = 2025

url = f"https://www.uefa.com/uefachampionsleague/history/seasons/{year_of_interest}/statistics/players/" 
driver.get(url)
headers = {'User-Agent': 'Mozilla/5.0'}

#### The following code block is the longest. It took between 5 - 10 minutes to run

In [3]:
# Let's have the browser scroll automatically
i=0

# The epoch number should be 70
epochs = 70 

with tqdm(total = epochs) as pbar:
    while i<=epochs:
        prev_page_source = driver.page_source
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(4) # This pauses the page for 4 seconds to let it load
        new_page_source = driver.page_source
        if new_page_source == prev_page_source:
            break # Stop scrolling if no new content is loaded
        i+=1
        pbar.update(1)

71it [05:34,  4.71s/it]                        


In [4]:
# Parse the HTML content of the page using BeautifulSoup
soup = BeautifulSoup(driver.page_source, "html.parser")

In [5]:
# This part converts the variable soup into a string and puts it into an html document called Players
with open(os.path.join(os.path.abspath(".."),"Players2.html"),'w' ,encoding= 'utf-8') as file:
    file.write(str(soup))

In [6]:
# Create a RegEx for Numerical Data Extraction
elements = soup.find_all(class_="ag-cell-value")

In [7]:
# This will give us a list of all the numerical data 
data_numbers = [element.text for element in elements]

# Since there are 6 columns, we can split the data every 6 numbers.
sublists = list(mit.chunked(data_numbers, 8))         

sublists

[['01', 'PachoParis - Defender', '1542', '17', '0', '0', '159.2', '33.8'],
 ['02', 'HakimiParis - Defender', '1540', '17', '4', '5', '176.8', '36.9'],
 ['03',
  'Nuno MendesParis - Defender',
  '1449',
  '16',
  '4',
  '2',
  '145.4',
  '35.7'],
 ['04', 'VitinhaParis - Midfielder', '1448', '17', '2', '2', '180.2', '32.3'],
 ['05', 'MarquinhosParis - Defender', '1439', '16', '0', '0', '158.1', '33.2'],
 ['06',
  'João NevesParis - Midfielder',
  '1435',
  '17',
  '1',
  '1',
  '184.3',
  '31.7'],
 ['07',
  'DonnarummaParis - Goalkeeper',
  '1380',
  '15',
  '0',
  '0',
  '71.7',
  '26.1'],
 ['08', 'SommerInter - Goalkeeper', '1290', '14', '0', '0', '50.5', '25.2'],
 ['09',
  'KimmichBayern München - Midfielder',
  '1260',
  '14',
  '0',
  '6',
  '178.9',
  '32.2'],
 ['10',
  'ValverdeReal Madrid - Midfielder',
  '1236',
  '14',
  '0',
  '3',
  '140',
  '35.2'],
 ['11',
  'RaphinhaBarcelona - Forward',
  '1225',
  '14',
  '13',
  '9',
  '126.8',
  '34.7'],
 ['12',
  'RüdigerReal Madrid -

### Now that we have the Data relatively structured, we can convert it into a dataframe

In [8]:
columns = ["Index", "Name, Team - Position ", "Minutes_played","Matches","Goals","Assists","Distance_covered","Top_speed"]
initial_data = pd.DataFrame(
    data = [i for i in sublists], 
    columns = columns,
    )

In [9]:
print(f'Initial Data Shape: {initial_data.shape}')
initial_data.head(5)

Initial Data Shape: (1107, 8)


,Index,"Name, Team - Position",Minutes_played,Matches,Goals,Assists,Distance_covered,Top_speed
0,01,PachoParis - Defender,1542,17,0,0,159.2,33.8
1,02,HakimiParis - Defender,1540,17,4,5,176.8,36.9
2,03,Nuno MendesParis - Defender,1449,16,4,2,145.4,35.7
3,04,VitinhaParis - Midfielder,1448,17,2,2,180.2,32.3
4,05,MarquinhosParis - Defender,1439,16,0,0,158.1,33.2


In [10]:
# Let's focus on columns 3 and onward
Numerical_Data = initial_data.iloc[:, 2:]

print(f'Numerical Data Shape: {Numerical_Data.shape}')
Numerical_Data.head(10)

Numerical Data Shape: (1107, 6)


,Minutes_played,Matches,Goals,Assists,Distance_covered,Top_speed
0,1542,17,0,0,159.2,33.8
1,1540,17,4,5,176.8,36.9
2,1449,16,4,2,145.4,35.7
3,1448,17,2,2,180.2,32.3
4,1439,16,0,0,158.1,33.2
5,1435,17,1,1,184.3,31.7
6,1380,15,0,0,71.7,26.1
7,1290,14,0,0,50.5,25.2
8,1260,14,0,6,178.9,32.2
9,1236,14,0,3,140,35.2


### Great, now we have the numerical Data. Now, we'll take care of the corresponding players and their teams

In [11]:
# Let's create some lists for our DataFrame
player_name, team = [], []

for pk_identifier in soup.find_all(class_='pk-identifier'):
   
        primary_span   = pk_identifier.find('span', slot='primary')
        secondary_span = pk_identifier.find('span', slot='secondary')

        if primary_span and secondary_span:
            primary_text   = primary_span.get_text(strip=True)
            secondary_text = secondary_span.get_text(strip = True)

        # Concatenate with a space in between
        player_name.append(primary_text)
        team.append(secondary_text)

In [12]:
player_name

['Pacho',
 'Hakimi',
 'Nuno Mendes',
 'Vitinha',
 'Marquinhos',
 'João Neves',
 'Donnarumma',
 'Sommer',
 'Kimmich',
 'Valverde',
 'Raphinha',
 'Rüdiger',
 'Raya',
 'Kobel',
 'Dembélé',
 'Fabián Ruiz',
 'Bellingham',
 'Mbappé',
 'Kane',
 'Pedri',
 'Courtois',
 'Koundé',
 'Vinícius Júnior',
 'Yamal',
 'Barcola',
 'Guirassy',
 'Vanaken',
 'Hancko',
 'Otamendi',
 'Schlotterbeck',
 'Mechele',
 'Mignolet',
 'E. Martínez',
 'Trubin',
 'De Cuyper',
 'Bastoni',
 'Kim',
 'Ordoñez',
 'Tielemans',
 'Olise',
 'Rogers',
 'Benítez',
 'Rice',
 'Orkun Kökçü',
 'Barella',
 'Aursnes',
 'Konsa',
 'Upamecano',
 'Saliba',
 'Lewandowski',
 'Cubarsí',
 'Dumfries',
 'Iñigo Martínez',
 'Ryerson',
 'Wellenreuther',
 'M. Thuram',
 'Jashari',
 'Can',
 'Tzolis',
 'Onyedika',
 'Martinelli',
 'Oblak',
 'Tomás Araújo',
 'Paixão',
 'Gross',
 'Boscagli',
 'Pavard',
 'Pavlidis',
 'L. de Jong',
 'Carreras',
 'Alexsandro',
 'McGregor',
 'Tah',
 'Schmeichel',
 'Chevalier',
 'Partey',
 'Mendy',
 'Brandt',
 'Rodrygo',
 'De R

In [13]:
team

['Paris - Defender',
 'Paris - Defender',
 'Paris - Defender',
 'Paris - Midfielder',
 'Paris - Defender',
 'Paris - Midfielder',
 'Paris - Goalkeeper',
 'Inter - Goalkeeper',
 'Bayern München - Midfielder',
 'Real Madrid - Midfielder',
 'Barcelona - Forward',
 'Real Madrid - Defender',
 'Arsenal - Goalkeeper',
 'B. Dortmund - Goalkeeper',
 'Paris - Forward',
 'Paris - Midfielder',
 'Real Madrid - Midfielder',
 'Real Madrid - Forward',
 'Bayern München - Forward',
 'Barcelona - Midfielder',
 'Real Madrid - Goalkeeper',
 'Barcelona - Defender',
 'Real Madrid - Forward',
 'Barcelona - Forward',
 'Paris - Forward',
 'B. Dortmund - Forward',
 'Club Brugge - Midfielder',
 'Feyenoord - Defender',
 'Benfica - Defender',
 'B. Dortmund - Defender',
 'Club Brugge - Defender',
 'Club Brugge - Goalkeeper',
 'Aston Villa - Goalkeeper',
 'Benfica - Goalkeeper',
 'Club Brugge - Midfielder',
 'Inter - Defender',
 'Bayern München - Defender',
 'Club Brugge - Defender',
 'Aston Villa - Midfielder',
 'Ba

#### Let's Set Up a DataFrame for the Qualitative Information

In [14]:
data = pd.DataFrame({
        'Player_Name': player_name,
        'Team'       : team,
        'Position'   : team,
        })
data.shape

(1107, 3)

In [15]:
data.head(8)

,Player_Name,Team,Position
0,Pacho,Paris - Defender,Paris - Defender
1,Hakimi,Paris - Defender,Paris - Defender
2,Nuno Mendes,Paris - Defender,Paris - Defender
3,Vitinha,Paris - Midfielder,Paris - Midfielder
4,Marquinhos,Paris - Defender,Paris - Defender
5,João Neves,Paris - Midfielder,Paris - Midfielder
6,Donnarumma,Paris - Goalkeeper,Paris - Goalkeeper
7,Sommer,Inter - Goalkeeper,Inter - Goalkeeper


#### We should perform some error handling and processing

In [16]:
position_list = []

for index, item in enumerate(team):
    try:
        processed_item = item.split("-")[1].strip()
        position_list.append(processed_item)
    except Exception as e:
        processed_item = "!!!!!!!!!"
        position_list.append(processed_item)
        print(f"Error at index {index}: {e}, item: '{item}'")

In [17]:
len(position_list)

1107

In [18]:
# Let's make sure that Team and Position Columns are accurate 

Numerical_Data['Player_Name'] = np.array([i for i in player_name])
Numerical_Data['Team'] = np.array([i.split("-")[0].strip() for i in team])
Numerical_Data['Position'] = np.array([i for i in position_list])

C:\Users\18126\AppData\Local\Temp\ipykernel_26912\1375059972.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Numerical_Data['Player_Name'] = np.array([i for i in player_name])


In [19]:
Numerical_Data.columns

Index(['Minutes_played', 'Matches', 'Goals', 'Assists', 'Distance_covered',
       'Top_speed', 'Player_Name', 'Team', 'Position'],
      dtype='object')

In [20]:
# Let's compile both qualitative and quantitative data

Combined_Data = Numerical_Data[['Player_Name', 'Team', 'Position', 
                                'Minutes_played', 'Matches', 'Goals', 
                                'Assists', 'Distance_covered','Top_speed']]
Combined_Data.head()

,Player_Name,Team,Position,Minutes_played,Matches,Goals,Assists,Distance_covered,Top_speed
0,Pacho,Paris,Defender,1542,17,0,0,159.2,33.8
1,Hakimi,Paris,Defender,1540,17,4,5,176.8,36.9
2,Nuno Mendes,Paris,Defender,1449,16,4,2,145.4,35.7
3,Vitinha,Paris,Midfielder,1448,17,2,2,180.2,32.3
4,Marquinhos,Paris,Defender,1439,16,0,0,158.1,33.2


In [21]:
# Show rows where either Player Name, Team, or Position is empty

empty_player_name = Combined_Data[Combined_Data['Player_Name'].isna()]
empty_team        = Combined_Data[Combined_Data['Team'].isna()]
empty_position    = Combined_Data[Combined_Data['Position'].isna()]

print(f'Number of rows with an empty player name: {len(empty_player_name)}')
print(f'Number of rows with an empty team value: {len(empty_team)}')
print(f'Number of rows with an empty position value: {len(empty_position)}')

Number of rows with an empty player name: 0
Number of rows with an empty team value: 0
Number of rows with an empty position value: 0


In [22]:
# Let's see if there are any positions other than the only 4 that should be there

valid_positions = ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']
invalid_position_rows = Combined_Data[~Combined_Data['Position'].isin(valid_positions)]

invalid_position_rows

,Player_Name,Team,Position,Minutes_played,Matches,Goals,Assists,Distance_covered,Top_speed


In [23]:
# Let's Convert the names to ASCII format, which makes it more suitable for regression

Combined_Data['Player_Name'] = Combined_Data['Player_Name'].apply(unidecode.unidecode)
Combined_Data['Team'] = Combined_Data['Team'].apply(unidecode.unidecode)

In [24]:
Combined_Data.head(10)

,Player_Name,Team,Position,Minutes_played,Matches,Goals,Assists,Distance_covered,Top_speed
0,Pacho,Paris,Defender,1542,17,0,0,159.2,33.8
1,Hakimi,Paris,Defender,1540,17,4,5,176.8,36.9
2,Nuno Mendes,Paris,Defender,1449,16,4,2,145.4,35.7
3,Vitinha,Paris,Midfielder,1448,17,2,2,180.2,32.3
4,Marquinhos,Paris,Defender,1439,16,0,0,158.1,33.2
5,Joao Neves,Paris,Midfielder,1435,17,1,1,184.3,31.7
6,Donnarumma,Paris,Goalkeeper,1380,15,0,0,71.7,26.1
7,Sommer,Inter,Goalkeeper,1290,14,0,0,50.5,25.2
8,Kimmich,Bayern Munchen,Midfielder,1260,14,0,6,178.9,32.2
9,Valverde,Real Madrid,Midfielder,1236,14,0,3,140,35.2


#### We can save our model now

In [ ]:
# Let's create a Data folder to store the data in
Data_Folder= 'Data'

if not os.path.exists(Data_Folder):
    os.makedirs(Data_Folder)

# Now we can save the model in the newly created folder
file_path = os.path.join(Data_Folder, f'UEFA Stats {year_of_interest}.xlsx')

Combined_Data.to_excel(file_path, index=False, header=True)

print(f'The file was saved to {file_path}')
# Last Comment 12

The file was saved to Data\UEFA Stats 2025.xlsx
